# Bookeau · Calidad para demanda y satisfacción

Los dos análisis, sus preguntas y denominadores están definidos en `docs/analisis/definicion_analisis.md`. La evaluación de calidad está en `docs/calidad/evaluacion_para_analisis.md`.

Este cuaderno consulta los resultados de `src/preparar_analisis_bookeau.py`, que fue ejecutado sobre los 114 Excel. Las fuentes permanecen intactas. Las celdas de este cuaderno todavía no se han ejecutado en Jupyter.

No hay una definición confirmada de asistencia ni del criterio de encuesta Inválida en el README de origen. Los resultados de calificación usan explícitamente el subconjunto marcado Válida y el dominio observado 1–5, con el filtrado de validez adoptado como R02; los resultados siguen siendo exploratorios.

Actualización R02: el filtrado por Válida fue adoptado por el usuario. Las encuestas Inválida permanecen en Silver, pero se excluyen de indicadores y análisis de respuestas mediante `incluir_encuesta_en_analisis`. Los otros supuestos de comparabilidad siguen sujetos a revisión.


In [1]:
from pathlib import Path
import csv
from html import escape
from IPython.display import display, HTML

PROYECTO = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "docs" / "plan_entrega_1.md").is_file()
)
RESULTADOS = PROYECTO / "docs" / "calidad"

def cargar(nombre):
    with (RESULTADOS / nombre).open(newline="", encoding="utf-8") as archivo:
        return list(csv.DictReader(archivo))

def tabla(filas, columnas=None, limite=None):
    if not filas:
        print("Sin registros para esta selección.")
        return
    columnas = columnas or list(filas[0])
    mostrar = filas if limite is None else filas[:limite]
    cabecera = "".join(f"<th>{escape(c)}</th>" for c in columnas)
    cuerpo = "".join(
        "<tr>" + "".join(f"<td>{escape(str(fila[c]))}</td>" for c in columnas) + "</tr>"
        for fila in mostrar
    )
    display(HTML(f"<table><thead><tr>{cabecera}</tr></thead><tbody>{cuerpo}</tbody></table>"))
    print(f"Mostrando {len(mostrar)} de {len(filas)} registros.")


## 1. Eventos de reserva y agenda

Los conteos incluyen espera, cancelaciones y otros estados. No equivalen a tutorías atendidas. La hora corresponde al inicio programado tal como se exporta; su zona horaria no está confirmada.


In [2]:
demanda = cargar("demanda_por_periodo_modalidad_servicio_estado.csv")
PERIODO = "202610"
MODALIDAD = "Express"
tabla([r for r in demanda if r["periodo"] == PERIODO and r["tipo_horario_original"] == MODALIDAD])
agenda = cargar("agenda_por_dia_hora.csv")
tabla([r for r in agenda if r["periodo"] == PERIODO and r["tipo_horario_original"] == MODALIDAD], limite=30)


periodo,tipo_horario_original,servicio_original,estado_reserva_original,eventos_reserva
202610,Express,EDA,Cancelada a tiempo,10
202610,Express,EDA,En cola,3
202610,Express,EDA,En ejecución,3
202610,Express,EDA,Finalizada,12
202610,Express,EDA,No asistió,6
202610,Express,EDA,Realizada,2
202610,Express,IP,Cancelada a tiempo,158
202610,Express,IP,Cancelada con excusa,11
202610,Express,IP,Cola cancelada por reservación,4
202610,Express,IP,En cola,13


Mostrando 14 de 14 registros.


periodo,tipo_horario_original,servicio_original,dia_semana_iso,hora_inicio,eventos_reserva
202610,Express,EDA,1,16,4
202610,Express,EDA,3,13,9
202610,Express,EDA,4,12,4
202610,Express,EDA,4,13,8
202610,Express,EDA,5,12,4
202610,Express,EDA,5,13,7
202610,Express,IP,1,9,13
202610,Express,IP,1,10,9
202610,Express,IP,1,12,17
202610,Express,IP,1,13,10


Mostrando 30 de 36 registros.


## 2. Estados y casos para revisión

49 eventos presentan combinaciones de estado y llegada que ameritan revisión: 8 Finalizada sin llegada, 9 No asistió con llegada y 32 estados de cancelación con llegada. No se corrigen ni excluyen automáticamente.


In [3]:
tabla(cargar("catalogo_estados_pendiente.csv"))
casos = cargar("casos_revision_estado_llegada.csv")
from collections import Counter
motivos = Counter(r["motivo_revision"] for r in casos)
tabla([{"motivo": motivo, "casos": n} for motivo, n in motivos.items()])
# Para revisar registros concretos con el responsable de la fuente:
# tabla(casos)


estado_original,eventos,con_llegada,sin_llegada,tratamiento_actual,definicion_pendiente
Cancelada,1764,0,1764,Conservar etiqueta original; contabilizar por separado,"Regla operativa, si es terminal y pertenencia al denominador de asistencia/cancelación"
Cancelada a tiempo,18147,4,18143,Conservar etiqueta original; contabilizar por separado,"Regla operativa, si es terminal y pertenencia al denominador de asistencia/cancelación"
Cancelada con excusa,784,16,768,Conservar etiqueta original; contabilizar por separado,"Regla operativa, si es terminal y pertenencia al denominador de asistencia/cancelación"
Cancelada por sanción,2523,8,2515,Conservar etiqueta original; contabilizar por separado,"Regla operativa, si es terminal y pertenencia al denominador de asistencia/cancelación"
Cancelada y sancionada,286,0,286,Conservar etiqueta original; contabilizar por separado,"Regla operativa, si es terminal y pertenencia al denominador de asistencia/cancelación"
Cola cancelada por reservación,4603,4,4599,Conservar etiqueta original; contabilizar por separado,"Regla operativa, si es terminal y pertenencia al denominador de asistencia/cancelación"
En cola,10215,0,10215,Conservar etiqueta original; contabilizar por separado,"Regla operativa, si es terminal y pertenencia al denominador de asistencia/cancelación"
En ejecución,307,307,0,Conservar etiqueta original; contabilizar por separado,"Regla operativa, si es terminal y pertenencia al denominador de asistencia/cancelación"
Finalizada,38541,38533,8,Conservar original; campo analítico Atendida según R01,Diferencia técnica oficial y denominador elegible; equivalencia de trabajo aportada por María Alejandra
No asistió,5086,9,5077,Conservar etiqueta original; contabilizar por separado,"Regla operativa, si es terminal y pertenencia al denominador de asistencia/cancelación"


Mostrando 12 de 12 registros.


motivo,casos
Estado de cancelación con marca de llegada,32
Finalizada sin marca de llegada,8
No asistió con marca de llegada,9


Mostrando 3 de 3 registros.


## 3. Denominador de calificaciones según R02

El conteo utilizable se limita a respuestas marcadas Válida con calificación 1–5. Los otros estados y los faltantes se conservan en los resultados agregados. Completitud del ítem es la proporción de encuestas marcadas Válida que tiene calificación utilizable; no es tasa de respuesta entre todas las tutorías.


In [4]:
tabla(cargar("resumen_elegibilidad_satisfaccion.csv"))
satisfaccion = cargar("satisfaccion_por_periodo_modalidad_servicio.csv")
tabla([r for r in satisfaccion if r["periodo"] == PERIODO])


grupo,respuestas,marcadas_valida,otro_estado,calificaciones_validas_1_a_5,faltantes_en_validas,fuera_dominio_en_validas
Encuesta Express,6759,6706,53,4938,1768,0
Encuesta Normal,16379,16268,111,11233,5035,0


Mostrando 2 de 2 registros.


grupo_encuesta,periodo,tipo_horario_reserva,servicio_reserva,respuestas_exportadas,respuestas_marcadas_valida,respuestas_otro_estado,calificaciones_1_a_5_en_validas,sin_calificacion_en_validas,fuera_dominio_en_validas,completitud_item_en_validas_pct,media_descriptiva_provisional,mediana_descriptiva_provisional,n_calificacion_1,n_calificacion_2,n_calificacion_3,n_calificacion_4,n_calificacion_5
Encuesta Express,202610,Express,EDA,14,14,0,14,0,0,100.0,4.7857,5.0,0,0,0,3,11
Encuesta Express,202610,Express,IP,267,267,0,267,0,0,100.0,4.6891,5,1,2,10,53,201
Encuesta Normal,202610,Normal,EDA,29,28,1,28,0,0,100.0,4.8214,5.0,0,0,1,3,24
Encuesta Normal,202610,Normal,IP,632,627,5,627,0,0,100.0,4.7305,5,1,5,32,86,503
Encuesta Normal,202610,Normal pico,IP,23,23,0,23,0,0,100.0,4.6522,5,0,0,2,4,17


Mostrando 5 de 5 registros.


## 4. Estratos para comparar Express y Normal

Hay 37 combinaciones de período y servicio original con calificaciones en ambas modalidades exactas. Normal pico y modalidades Deprecated no se fusionan con Normal o Express. La igualdad de período, servicio y pregunta no garantiza equivalencia del formulario ni ausencia de sesgo de respuesta.


In [5]:
comparacion = cargar("estratos_comunes_express_normal.csv")
tabla(comparacion)


periodo,servicio_original,n_express,n_normal,media_express_provisional,media_normal_provisional,completitud_item_express_pct,completitud_item_normal_pct,limite
201819,"Deprecated, APO 1",10,31,4.8,4.9355,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201819,"Deprecated, APO 2",7,13,5,4.9231,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201820,"Deprecated, APO 1",490,1086,4.8551,4.8766,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201820,"Deprecated, APO 2",50,108,4.9,4.8148,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201820,"Deprecated, Apoyo",14,22,4.9286,4.9545,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201820,"Deprecated, IP Nivelación",1,1,5,5,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201910,"Deprecated, APO 1",507,1022,4.8107,4.8503,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201910,"Deprecated, APO 2",48,94,4.7708,4.7979,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201910,"Deprecated, Apoyo",10,3,5,4.6667,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201910,"Deprecated, IP Nivelación",11,14,5,4.7857,100.0,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"


Mostrando 37 de 37 registros.


## 5. Regenerar las tablas

El script vuelve a leer todas las fuentes y comprueba unicidad y vínculos antes de agregar. Si cambian fuentes o reglas, revisar también las definiciones y la narrativa de los documentos.


In [6]:
# import runpy
# import sys
# sys.path.insert(0, str(PROYECTO / "src"))
# runpy.run_path(str(PROYECTO / "src" / "preparar_analisis_bookeau.py"), run_name="__main__")
